<a href="https://colab.research.google.com/github/Tanwi20024/House_pricing_prediction/blob/main/Sentiment_Analysis_one.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install django textblob pyngrok -q
!python -m textblob.download_corpora -q
print("✅ All packages installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 37.4 MB/s eta 0:00:00
[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Unzipping corpora/brown.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package conll2000 to /root/nltk_data...
[nltk_data]   Unzipping corpora/conll2000.zip.
[nltk_data] Downloading package movie_reviews to /root/nltk_data...
[nltk_data]   Unzipping corpora/movie_reviews.zip.
Finished.
✅ All packages installed!


In [2]:
import shutil, os, subprocess

# Clean old project
if os.path.exists('/content/sentiment_project'):
    shutil.rmtree('/content/sentiment_project')
    print("🗑️ Old project removed!")

# Create fresh project
os.chdir('/content')
subprocess.run(['django-admin', 'startproject', 'sentiment_project'], check=True)
os.chdir('/content/sentiment_project')
subprocess.run(['python', 'manage.py', 'startapp', 'analyzer'], check=True)
os.makedirs('analyzer/templates/analyzer', exist_ok=True)
os.makedirs('analyzer/static/analyzer', exist_ok=True)

print("✅ Fresh project created!")
print("📁", os.listdir('.'))

✅ Fresh project created!
📁 ['sentiment_project', 'analyzer', 'manage.py']


In [3]:
import os
os.chdir('/content/sentiment_project')

with open('sentiment_project/settings.py', 'r') as f:
    s = f.read()

# Fix ALLOWED_HOSTS
s = s.replace("ALLOWED_HOSTS = []", "ALLOWED_HOSTS = ['*']")

# Disable CSRF middleware
s = s.replace(
    "'django.middleware.csrf.CsrfViewMiddleware',",
    "# 'django.middleware.csrf.CsrfViewMiddleware',  # Disabled for ngrok"
)

# Register analyzer app
s = s.replace(
    "'django.contrib.staticfiles',",
    "'django.contrib.staticfiles',\n    'analyzer',"
)

# Add CSRF trusted origins & static files config at end
s += """
# CSRF & ngrok settings
CSRF_TRUSTED_ORIGINS = ['https://*.ngrok-free.app', 'https://*.ngrok-free.dev', 'https://*.ngrok.io']
CSRF_COOKIE_SECURE = False
SESSION_COOKIE_SECURE = False

import os
STATIC_URL = '/static/'
STATICFILES_DIRS = [os.path.join(BASE_DIR, 'analyzer/static')]
"""

with open('sentiment_project/settings.py', 'w') as f:
    f.write(s)

print("✅ Settings configured!")

✅ Settings configured!


In [4]:
import os
os.chdir('/content/sentiment_project')

# Project-level urls.py
with open('sentiment_project/urls.py', 'w') as f:
    f.write("""
from django.contrib import admin
from django.urls import path, include

urlpatterns = [
    path('admin/', admin.site.urls),
    path('', include('analyzer.urls')),
]
""")

# App-level urls.py
with open('analyzer/urls.py', 'w') as f:
    f.write("""
from django.urls import path
from . import views

urlpatterns = [
    path('', views.home, name='home'),
    path('analyze/', views.analyze, name='analyze'),
    path('history/', views.history, name='history'),
]
""")

print("✅ URL routes created!")

✅ URL routes created!


In [5]:
import os
os.chdir('/content/sentiment_project')

with open('analyzer/views.py', 'w') as f:
    f.write("""
from django.shortcuts import render
from django.views.decorators.csrf import csrf_exempt
from django.http import JsonResponse
from textblob import TextBlob
import json

# In-memory history store
analysis_history = []

def home(request):
    return render(request, 'analyzer/index.html', {'history': analysis_history[-5:][::-1]})

@csrf_exempt
def analyze(request):
    if request.method == 'POST':
        try:
            data = json.loads(request.body)
            text = data.get('text', '').strip()
        except:
            text = request.POST.get('text', '').strip()

        if not text:
            return JsonResponse({'error': 'No text provided'}, status=400)

        blob = TextBlob(text)
        polarity    = round(blob.sentiment.polarity, 4)
        subjectivity = round(blob.sentiment.subjectivity, 4)

        if polarity > 0.1:
            sentiment = 'Positive'
            emoji     = '😊'
            color     = '#10b981'
        elif polarity < -0.1:
            sentiment = 'Negative'
            emoji     = '😞'
            color     = '#ef4444'
        else:
            sentiment = 'Neutral'
            emoji     = '😐'
            color     = '#f59e0b'

        # Polarity percentage (0-100)
        polarity_pct    = round((polarity + 1) / 2 * 100, 1)
        subjectivity_pct = round(subjectivity * 100, 1)

        # Word & sentence count
        words     = len(text.split())
        sentences = len(blob.sentences)

        result = {
            'text': text,
            'sentiment': sentiment,
            'emoji': emoji,
            'color': color,
            'polarity': polarity,
            'subjectivity': subjectivity,
            'polarity_pct': polarity_pct,
            'subjectivity_pct': subjectivity_pct,
            'words': words,
            'sentences': sentences,
        }

        # Save to history
        analysis_history.append(result)
        if len(analysis_history) > 20:
            analysis_history.pop(0)

        return JsonResponse(result)

    return JsonResponse({'error': 'Invalid method'}, status=405)

@csrf_exempt
def history(request):
    return JsonResponse({'history': analysis_history[-10:][::-1]})
""")

print("✅ Views created!")

✅ Views created!


In [6]:
import os
os.chdir('/content/sentiment_project')

html = r"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8"/>
<meta name="viewport" content="width=device-width, initial-scale=1.0"/>
<title>SentimentAI — Emotion Analyzer</title>
<link href="https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700;800&family=Playfair+Display:wght@600;700&display=swap" rel="stylesheet"/>
<link href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.4.0/css/all.min.css" rel="stylesheet"/>
<style>
  :root {
    --pink:    #f9a8d4;
    --purple:  #c084fc;
    --blue:    #93c5fd;
    --mint:    #6ee7b7;
    --peach:   #fcd34d;
    --bg:      #fdf4ff;
    --card:    rgba(255,255,255,0.75);
    --text:    #3b1f5e;
    --sub:     #7c6a91;
    --border:  rgba(192,132,252,0.25);
    --shadow:  0 8px 32px rgba(139,92,246,0.12);
  }

  * { margin:0; padding:0; box-sizing:border-box; }

  body {
    font-family: 'Inter', sans-serif;
    background: var(--bg);
    color: var(--text);
    min-height: 100vh;
    overflow-x: hidden;
  }

  /* ── Animated Background ── */
  .bg-blobs {
    position: fixed; inset: 0; z-index: 0; pointer-events: none; overflow: hidden;
  }
  .blob {
    position: absolute; border-radius: 50%; filter: blur(80px); opacity: 0.35;
    animation: float 12s ease-in-out infinite;
  }
  .blob-1 { width:500px; height:500px; background:#f9a8d4; top:-100px; left:-100px; animation-delay:0s; }
  .blob-2 { width:400px; height:400px; background:#c084fc; top:30%; right:-80px; animation-delay:3s; }
  .blob-3 { width:350px; height:350px; background:#93c5fd; bottom:-80px; left:20%; animation-delay:6s; }
  .blob-4 { width:300px; height:300px; background:#6ee7b7; bottom:20%; right:25%; animation-delay:9s; }

  @keyframes float {
    0%,100% { transform: translateY(0) scale(1); }
    50%      { transform: translateY(-30px) scale(1.05); }
  }

  /* ── Layout ── */
  .wrapper { position:relative; z-index:1; max-width:900px; margin:0 auto; padding:40px 20px 80px; }

  /* ── Header ── */
  header {
    text-align: center; margin-bottom: 48px;
  }
  .logo-badge {
    display: inline-flex; align-items: center; gap: 8px;
    background: rgba(255,255,255,0.8); border: 1px solid var(--border);
    backdrop-filter: blur(12px); border-radius: 50px;
    padding: 8px 20px; font-size: 0.82rem; font-weight: 600;
    color: var(--sub); margin-bottom: 20px;
    box-shadow: 0 2px 12px rgba(192,132,252,0.15);
  }
  .logo-badge i { color: var(--purple); }

  header h1 {
    font-family: 'Playfair Display', serif;
    font-size: clamp(2.2rem, 5vw, 3.4rem);
    background: linear-gradient(135deg, #9333ea, #ec4899, #3b82f6);
    -webkit-background-clip: text; -webkit-text-fill-color: transparent;
    background-clip: text; line-height: 1.2; margin-bottom: 14px;
  }
  header p {
    color: var(--sub); font-size: 1.05rem; font-weight: 400; max-width: 520px; margin: 0 auto;
  }

  /* ── Glass Card ── */
  .card {
    background: var(--card);
    backdrop-filter: blur(20px);
    border: 1px solid var(--border);
    border-radius: 24px;
    padding: 36px;
    box-shadow: var(--shadow);
    margin-bottom: 24px;
  }

  /* ── Input Area ── */
  .input-label {
    font-size: 0.85rem; font-weight: 600; color: var(--sub);
    text-transform: uppercase; letter-spacing: 1.5px; margin-bottom: 12px;
    display: flex; align-items: center; gap: 8px;
  }
  .input-label i { color: var(--purple); }

  .textarea-wrap { position: relative; }

  textarea {
    width: 100%; min-height: 160px;
    padding: 18px 20px; padding-bottom: 50px;
    background: rgba(255,255,255,0.9);
    border: 2px solid var(--border);
    border-radius: 16px; resize: vertical;
    font-family: 'Inter', sans-serif;
    font-size: 1rem; color: var(--text);
    outline: none; transition: border-color 0.3s, box-shadow 0.3s;
    line-height: 1.6;
  }
  textarea::placeholder { color: #b8a9c8; }
  textarea:focus {
    border-color: var(--purple);
    box-shadow: 0 0 0 4px rgba(192,132,252,0.15);
  }

  .char-count {
    position: absolute; bottom: 14px; right: 16px;
    font-size: 0.78rem; color: #b8a9c8; font-weight: 500;
  }

  /* ── Quick Chips ── */
  .chips { display: flex; flex-wrap: wrap; gap: 8px; margin: 16px 0; }
  .chip {
    padding: 6px 14px; border-radius: 50px; font-size: 0.8rem; font-weight: 500;
    cursor: pointer; border: 1px solid var(--border);
    background: rgba(255,255,255,0.8); color: var(--sub);
    transition: all 0.2s;
  }
  .chip:hover { background: var(--purple); color: white; border-color: var(--purple); transform: translateY(-1px); }
  .chip.pos { border-color: #10b981; color: #10b981; }
  .chip.neg { border-color: #ef4444; color: #ef4444; }
  .chip.neu { border-color: #f59e0b; color: #f59e0b; }
  .chip.pos:hover { background: #10b981; color: white; }
  .chip.neg:hover { background: #ef4444; color: white; }
  .chip.neu:hover { background: #f59e0b; color: white; }

  /* ── Analyze Button ── */
  .btn-wrap { margin-top: 20px; }
  .btn-analyze {
    width: 100%; padding: 16px;
    background: linear-gradient(135deg, #9333ea, #ec4899);
    color: white; border: none; border-radius: 14px;
    font-family: 'Inter', sans-serif;
    font-size: 1.05rem; font-weight: 700;
    cursor: pointer; letter-spacing: 0.5px;
    transition: all 0.3s; position: relative; overflow: hidden;
    box-shadow: 0 4px 20px rgba(147,51,234,0.35);
  }
  .btn-analyze:hover { transform: translateY(-2px); box-shadow: 0 8px 28px rgba(147,51,234,0.45); }
  .btn-analyze:active { transform: translateY(0); }
  .btn-analyze.loading { pointer-events: none; opacity: 0.8; }
  .btn-analyze i { margin-right: 8px; }

  /* ── Result Card ── */
  .result-card {
    display: none;
    animation: slideUp 0.5s cubic-bezier(0.34,1.56,0.64,1);
  }
  .result-card.show { display: block; }

  @keyframes slideUp {
    from { opacity:0; transform:translateY(30px) scale(0.97); }
    to   { opacity:1; transform:translateY(0) scale(1); }
  }

  /* Sentiment Banner */
  .sentiment-banner {
    border-radius: 18px; padding: 28px 32px;
    margin-bottom: 24px; text-align: center;
    position: relative; overflow: hidden;
  }
  .sentiment-banner::before {
    content: ''; position: absolute; inset: 0;
    opacity: 0.08; border-radius: 18px;
  }
  .banner-emoji { font-size: 3.5rem; margin-bottom: 10px; display: block; animation: bounce 1s ease; }
  @keyframes bounce { 0%,100%{transform:scale(1)} 50%{transform:scale(1.2)} }
  .banner-label {
    font-family: 'Playfair Display', serif;
    font-size: 2rem; font-weight: 700; margin-bottom: 6px;
  }
  .banner-sub { font-size: 0.9rem; opacity: 0.75; font-weight: 500; }

  .banner-pos { background: linear-gradient(135deg, #d1fae5, #a7f3d0); color: #065f46; }
  .banner-neg { background: linear-gradient(135deg, #fee2e2, #fecaca); color: #7f1d1d; }
  .banner-neu { background: linear-gradient(135deg, #fef3c7, #fde68a); color: #78350f; }

  /* Score Grid */
  .score-grid { display: grid; grid-template-columns: 1fr 1fr; gap: 16px; margin-bottom: 24px; }

  .score-item {
    background: rgba(255,255,255,0.9); border: 1px solid var(--border);
    border-radius: 16px; padding: 20px; text-align: center;
    transition: transform 0.2s;
  }
  .score-item:hover { transform: translateY(-3px); }
  .score-icon { font-size: 1.5rem; margin-bottom: 8px; }
  .score-name { font-size: 0.75rem; font-weight: 600; color: var(--sub); text-transform: uppercase; letter-spacing: 1px; margin-bottom: 6px; }
  .score-num  { font-size: 2rem; font-weight: 800; margin-bottom: 4px; }
  .score-desc { font-size: 0.72rem; color: #b8a9c8; }

  /* Progress Bars */
  .progress-section { margin-bottom: 24px; }
  .prog-row { margin-bottom: 16px; }
  .prog-header { display: flex; justify-content: space-between; align-items: center; margin-bottom: 8px; }
  .prog-title { font-size: 0.82rem; font-weight: 600; color: var(--sub); display: flex; align-items: center; gap: 6px; }
  .prog-val   { font-size: 0.85rem; font-weight: 700; color: var(--text); }
  .prog-track {
    height: 10px; background: rgba(192,132,252,0.1);
    border-radius: 50px; overflow: hidden;
  }
  .prog-fill {
    height: 100%; border-radius: 50px;
    transition: width 1.2s cubic-bezier(0.34,1.56,0.64,1);
    width: 0;
  }
  .prog-pol  { background: linear-gradient(90deg, #9333ea, #ec4899); }
  .prog-subj { background: linear-gradient(90deg, #3b82f6, #06b6d4); }

  /* Stats Row */
  .stats-row { display: flex; gap: 12px; }
  .stat-pill {
    flex: 1; background: rgba(255,255,255,0.8); border: 1px solid var(--border);
    border-radius: 12px; padding: 14px; text-align: center;
  }
  .stat-pill .num { font-size: 1.4rem; font-weight: 800; color: var(--purple); }
  .stat-pill .lbl { font-size: 0.72rem; color: var(--sub); font-weight: 500; margin-top: 2px; }

  /* Analyzed Text */
  .analyzed-quote {
    background: rgba(255,255,255,0.7); border-left: 4px solid var(--purple);
    border-radius: 0 12px 12px 0; padding: 16px 20px; margin-top: 20px;
    font-style: italic; color: var(--sub); font-size: 0.92rem; line-height: 1.6;
    position: relative;
  }
  .analyzed-quote::before {
    content: '"'; font-size: 3rem; color: var(--purple); opacity: 0.2;
    position: absolute; top: -8px; left: 10px; font-family: serif; line-height: 1;
  }

  /* History */
  .section-title {
    font-family: 'Playfair Display', serif;
    font-size: 1.3rem; color: var(--text); margin-bottom: 16px;
    display: flex; align-items: center; gap: 10px;
  }
  .section-title i { color: var(--purple); font-size: 1rem; }

  .history-list { display: flex; flex-direction: column; gap: 10px; }
  .history-item {
    background: rgba(255,255,255,0.8); border: 1px solid var(--border);
    border-radius: 14px; padding: 14px 18px;
    display: flex; align-items: center; gap: 14px;
    transition: transform 0.2s, box-shadow 0.2s; cursor: default;
  }
  .history-item:hover { transform: translateX(4px); box-shadow: var(--shadow); }
  .hist-emoji { font-size: 1.6rem; flex-shrink: 0; }
  .hist-body  { flex: 1; min-width: 0; }
  .hist-text  { font-size: 0.88rem; color: var(--text); white-space: nowrap; overflow: hidden; text-overflow: ellipsis; margin-bottom: 3px; }
  .hist-meta  { font-size: 0.75rem; color: var(--sub); }
  .hist-badge {
    padding: 4px 12px; border-radius: 50px; font-size: 0.72rem; font-weight: 700;
    flex-shrink: 0;
  }
  .badge-pos { background: #d1fae5; color: #065f46; }
  .badge-neg { background: #fee2e2; color: #7f1d1d; }
  .badge-neu { background: #fef3c7; color: #78350f; }

  /* Empty state */
  .empty-state { text-align: center; padding: 40px; color: var(--sub); }
  .empty-state i { font-size: 2.5rem; margin-bottom: 12px; opacity: 0.3; display: block; }

  /* Footer */
  footer {
    text-align: center; margin-top: 48px;
    color: var(--sub); font-size: 0.82rem;
  }
  footer span { color: var(--purple); }

  /* Spinner */
  .spinner {
    display: inline-block; width: 18px; height: 18px;
    border: 2px solid rgba(255,255,255,0.4);
    border-top-color: white; border-radius: 50%;
    animation: spin 0.7s linear infinite; vertical-align: middle; margin-right: 8px;
  }
  @keyframes spin { to { transform: rotate(360deg); } }

  @media(max-width:600px) {
    .score-grid { grid-template-columns: 1fr 1fr; }
    .stats-row  { flex-wrap: wrap; }
    .card { padding: 24px; }
    header h1 { font-size: 2rem; }
  }
</style>
</head>
<body>

<!-- Animated background blobs -->
<div class="bg-blobs">
  <div class="blob blob-1"></div>
  <div class="blob blob-2"></div>
  <div class="blob blob-3"></div>
  <div class="blob blob-4"></div>
</div>

<div class="wrapper">

  <!-- Header -->
  <header>
    <div class="logo-badge"><i class="fas fa-brain"></i> Powered by TextBlob NLP</div>
    <h1>Emotion Intelligence<br>at Your Fingertips</h1>
    <p>Uncover the sentiment, polarity & subjectivity hidden in any text — instantly.</p>
  </header>

  <!-- Input Card -->
  <div class="card">
    <div class="input-label"><i class="fas fa-pen-nib"></i> Enter your text</div>

    <div class="textarea-wrap">
      <textarea id="textInput" placeholder="Write something... a review, a tweet, a journal entry, anything you want to analyze ✨" oninput="updateCount()"></textarea>
      <span class="char-count" id="charCount">0 chars</span>
    </div>

    <!-- Quick test chips -->
    <div style="margin-top:14px;">
      <div class="input-label" style="margin-bottom:8px;"><i class="fas fa-bolt"></i> Try a sample</div>
      <div class="chips">
        <span class="chip pos" onclick="setChip(this)">😊 I absolutely love this!</span>
        <span class="chip neg" onclick="setChip(this)">😞 This is terrible and disappointing.</span>
        <span class="chip neu" onclick="setChip(this)">😐 The event starts at 9am on Monday.</span>
        <span class="chip pos" onclick="setChip(this)">🌟 What a wonderful experience!</span>
        <span class="chip neg" onclick="setChip(this)">💔 I deeply regret this decision.</span>
      </div>
    </div>

    <div class="btn-wrap">
      <button class="btn-analyze" id="analyzeBtn" onclick="analyze()">
        <i class="fas fa-magic"></i> Analyze Sentiment
      </button>
    </div>
  </div>

  <!-- Result Card -->
  <div class="card result-card" id="resultCard">

    <!-- Sentiment Banner -->
    <div class="sentiment-banner" id="sentBanner">
      <span class="banner-emoji" id="sentEmoji"></span>
      <div class="banner-label" id="sentLabel"></div>
      <div class="banner-sub"  id="sentSub"></div>
    </div>

    <!-- Scores -->
    <div class="score-grid">
      <div class="score-item">
        <div class="score-icon">📊</div>
        <div class="score-name">Polarity</div>
        <div class="score-num" id="polarityNum" style="color:#9333ea">—</div>
        <div class="score-desc">-1.0 negative → +1.0 positive</div>
      </div>
      <div class="score-item">
        <div class="score-icon">🎭</div>
        <div class="score-name">Subjectivity</div>
        <div class="score-num" id="subjectivityNum" style="color:#3b82f6">—</div>
        <div class="score-desc">0.0 objective → 1.0 subjective</div>
      </div>
    </div>

    <!-- Progress Bars -->
    <div class="progress-section">
      <div class="prog-row">
        <div class="prog-header">
          <span class="prog-title"><i class="fas fa-chart-line" style="color:#9333ea"></i> Polarity Strength</span>
          <span class="prog-val" id="polPct">—</span>
        </div>
        <div class="prog-track"><div class="prog-fill prog-pol" id="polBar"></div></div>
      </div>
      <div class="prog-row">
        <div class="prog-header">
          <span class="prog-title"><i class="fas fa-theater-masks" style="color:#3b82f6"></i> Subjectivity Level</span>
          <span class="prog-val" id="subjPct">—</span>
        </div>
        <div class="prog-track"><div class="prog-fill prog-subj" id="subjBar"></div></div>
      </div>
    </div>

    <!-- Word/Sentence stats -->
    <div class="stats-row">
      <div class="stat-pill">
        <div class="num" id="wordCount">—</div>
        <div class="lbl">Words</div>
      </div>
      <div class="stat-pill">
        <div class="num" id="sentCount">—</div>
        <div class="lbl">Sentences</div>
      </div>
      <div class="stat-pill">
        <div class="num" id="charCountStat">—</div>
        <div class="lbl">Characters</div>
      </div>
    </div>

    <!-- Analyzed quote -->
    <div class="analyzed-quote" id="analyzedQuote"></div>
  </div>

  <!-- History -->
  <div class="card" id="historyCard">
    <div class="section-title"><i class="fas fa-history"></i> Recent Analyses</div>
    <div class="history-list" id="historyList">
      <div class="empty-state">
        <i class="fas fa-inbox"></i>
        No analyses yet. Try entering some text above!
      </div>
    </div>
  </div>

  <footer>
    Built with <span>♥</span> using Django & TextBlob &nbsp;·&nbsp; SentimentAI
  </footer>
</div>

<script>
  const ANALYZE_URL = '/analyze/';

  function updateCount() {
    const t = document.getElementById('textInput').value;
    document.getElementById('charCount').textContent = t.length + ' chars';
  }

  function setChip(el) {
    const text = el.textContent.replace(/^[^\s]+\s/, '').trim();
    document.getElementById('textInput').value = text;
    updateCount();
    document.getElementById('textInput').focus();
  }

  async function analyze() {
    const text = document.getElementById('textInput').value.trim();
    if (!text) {
      document.getElementById('textInput').style.borderColor = '#ef4444';
      setTimeout(() => document.getElementById('textInput').style.borderColor = '', 1500);
      return;
    }

    const btn = document.getElementById('analyzeBtn');
    btn.classList.add('loading');
    btn.innerHTML = '<span class="spinner"></span> Analyzing...';

    try {
      const res = await fetch(ANALYZE_URL, {
        method: 'POST',
        headers: { 'Content-Type': 'application/json' },
        body: JSON.stringify({ text })
      });
      const data = await res.json();
      showResult(data);
      addHistory(data);
    } catch(e) {
      alert('Something went wrong. Please try again.');
    }

    btn.classList.remove('loading');
    btn.innerHTML = '<i class="fas fa-magic"></i> Analyze Again';
  }

  function showResult(d) {
    const card = document.getElementById('resultCard');
    card.classList.add('show');
    card.scrollIntoView({ behavior: 'smooth', block: 'start' });

    // Banner
    const banner = document.getElementById('sentBanner');
    banner.className = 'sentiment-banner banner-' + d.sentiment.toLowerCase().slice(0,3);
    document.getElementById('sentEmoji').textContent = d.emoji;
    document.getElementById('sentLabel').textContent = d.sentiment;
    document.getElementById('sentSub').textContent =
      d.sentiment === 'Positive' ? 'The text carries a positive, uplifting tone.' :
      d.sentiment === 'Negative' ? 'The text reflects a negative or critical tone.' :
      'The text appears balanced and factual in tone.';

    // Scores
    document.getElementById('polarityNum').textContent = d.polarity;
    document.getElementById('subjectivityNum').textContent = d.subjectivity;

    // Progress bars
    setTimeout(() => {
      document.getElementById('polBar').style.width = d.polarity_pct + '%';
      document.getElementById('subjBar').style.width = d.subjectivity_pct + '%';
    }, 100);

    document.getElementById('polPct').textContent  = d.polarity_pct + '%';
    document.getElementById('subjPct').textContent = d.subjectivity_pct + '%';

    // Stats
    document.getElementById('wordCount').textContent   = d.words;
    document.getElementById('sentCount').textContent   = d.sentences;
    document.getElementById('charCountStat').textContent = d.text.length;

    // Quote
    document.getElementById('analyzedQuote').textContent = d.text;
  }

  function addHistory(d) {
    const list = document.getElementById('historyList');
    const empty = list.querySelector('.empty-state');
    if (empty) empty.remove();

    const badgeClass = d.sentiment === 'Positive' ? 'badge-pos' : d.sentiment === 'Negative' ? 'badge-neg' : 'badge-neu';

    const item = document.createElement('div');
    item.className = 'history-item';
    item.innerHTML = `
      <span class="hist-emoji">${d.emoji}</span>
      <div class="hist-body">
        <div class="hist-text">${d.text}</div>
        <div class="hist-meta">Polarity: ${d.polarity} &nbsp;·&nbsp; Subjectivity: ${d.subjectivity}</div>
      </div>
      <span class="hist-badge ${badgeClass}">${d.sentiment}</span>
    `;

    list.insertBefore(item, list.firstChild);

    // Keep max 8 in UI
    while (list.children.length > 8) list.removeChild(list.lastChild);
  }

  // Enter key to analyze
  document.addEventListener('keydown', e => {
    if (e.key === 'Enter' && e.ctrlKey) analyze();
  });
</script>
</body>
</html>"""

with open('analyzer/templates/analyzer/index.html', 'w') as f:
    f.write(html)

print("✅ Aesthetic HTML template created!")

✅ Aesthetic HTML template created!


In [8]:
from pyngrok import ngrok
ngrok.set_auth_token("3FXLhvaln5iTya3X3Z3vwe2J4DX_Sayre9AxxGMe8WLwNU9W")
print("✅ Ngrok authenticated!")

✅ Ngrok authenticated!


In [15]:
import os, threading, time
from pyngrok import ngrok

# Ensure any ngrok processes are killed at the very start to prevent conflicts
ngrok.kill()

os.chdir('/content/sentiment_project')

print("⏳ Running migrations...")
os.system("python manage.py migrate -q")
print("✅ Migrations done!")

def run_django():
    # It's important that this runs in the background and doesn't block
    os.system("python manage.py runserver 8000 --noreload")

t = threading.Thread(target=run_django, daemon=True)
t.start()

print("⏳ Starting server...")
time.sleep(6) # Give Django server some time to start

# Retry mechanism for ngrok connection
max_retries = 5
retry_count = 0
url = None

while retry_count < max_retries:
    try:
        # Ensure any *previously started* ngrok client is killed before attempting a new connection.
        # This handles cases where a previous ngrok.connect() failed but left a process running.
        # ngrok.kill() at the beginning of the cell should cover the initial state.
        ngrok.kill()
        time.sleep(2) # Give some time for the killed process to clean up

        # Attempt to establish a new ngrok tunnel.
        # Pyngrok will automatically try to get an ephemeral URL for free accounts.
        # We are *not* calling get_tunnels() and disconnect() here, as ngrok.kill()
        # ensures no local client is running, so querying it would be moot.
        # The 'endpoint already online' error is a server-side issue.
        url = ngrok.connect(8000)

        # If connection is successful, break the loop
        break
    except Exception as e:
        print(f"Ngrok connection failed: {e}")
        retry_count += 1
        if retry_count < max_retries:
            print(f"Retrying ngrok connection... (Attempt {retry_count}/{max_retries})")
            time.sleep(10) # Increased wait time to give ngrok server more time to clear
        else:
            print("Maximum retries reached. Could not establish ngrok tunnel.")

if url:
    print("=" * 55)
    print(f"🌸 Your App is LIVE at: {url}")
    print("=" * 55)
else:
    print("=" * 55)
    print("❌ Failed to establish ngrok tunnel. Please check the logs above for details.")
    print("=" * 55)

⏳ Running migrations...
✅ Migrations done!
⏳ Starting server...


Ngrok connection failed: ngrok client exception, API returned 502: {"error_code":103,"status_code":502,"msg":"failed to start tunnel","details":{"err":"failed to start tunnel: The endpoint 'https://detector-reemerge-blip.ngrok-free.dev' is already online. Either\n1. stop your existing endpoint first, or\n2. start both endpoints with `--pooling-enabled` to load balance between them.\r\n\r\nERR_NGROK_334\r\n"}}

Retrying ngrok connection... (Attempt 1/5)


Ngrok connection failed: ngrok client exception, API returned 502: {"error_code":103,"status_code":502,"msg":"failed to start tunnel","details":{"err":"failed to start tunnel: The endpoint 'https://detector-reemerge-blip.ngrok-free.dev' is already online. Either\n1. stop your existing endpoint first, or\n2. start both endpoints with `--pooling-enabled` to load balance between them.\r\n\r\nERR_NGROK_334\r\n"}}

Retrying ngrok connection... (Attempt 2/5)


Ngrok connection failed: ngrok client exception, API returned 502: {"error_code":103,"status_code":502,"msg":"failed to start tunnel","details":{"err":"failed to start tunnel: The endpoint 'https://detector-reemerge-blip.ngrok-free.dev' is already online. Either\n1. stop your existing endpoint first, or\n2. start both endpoints with `--pooling-enabled` to load balance between them.\r\n\r\nERR_NGROK_334\r\n"}}

Retrying ngrok connection... (Attempt 3/5)


Ngrok connection failed: ngrok client exception, API returned 502: {"error_code":103,"status_code":502,"msg":"failed to start tunnel","details":{"err":"failed to start tunnel: The endpoint 'https://detector-reemerge-blip.ngrok-free.dev' is already online. Either\n1. stop your existing endpoint first, or\n2. start both endpoints with `--pooling-enabled` to load balance between them.\r\n\r\nERR_NGROK_334\r\n"}}

Retrying ngrok connection... (Attempt 4/5)


Ngrok connection failed: ngrok client exception, API returned 502: {"error_code":103,"status_code":502,"msg":"failed to start tunnel","details":{"err":"failed to start tunnel: The endpoint 'https://detector-reemerge-blip.ngrok-free.dev' is already online. Either\n1. stop your existing endpoint first, or\n2. start both endpoints with `--pooling-enabled` to load balance between them.\r\n\r\nERR_NGROK_334\r\n"}}

Maximum retries reached. Could not establish ngrok tunnel.
❌ Failed to establish ngrok tunnel. Please check the logs above for details.


In [16]:
import os, threading, time, requests
from pyngrok import ngrok, conf

# Force kill all ngrok processes
ngrok.kill()
os.system("pkill -f ngrok")
time.sleep(3)

os.chdir('/content/sentiment_project')

# Start Django
def run_django():
    os.system("python manage.py runserver 8000 --noreload")

t = threading.Thread(target=run_django, daemon=True)
t.start()
time.sleep(5)

# Connect with fresh tunnel
try:
    url = ngrok.connect(8000)
    print("=" * 55)
    print(f"🌸 Your App is LIVE at: {url}")
    print("=" * 55)
except Exception as e:
    print(f"❌ Error: {e}")
    print("👉 Please go to https://dashboard.ngrok.com/endpoints and delete the active endpoint, then run this cell again.")

🌸 Your App is LIVE at: NgrokTunnel: "https://detector-reemerge-blip.ngrok-free.dev" -> "http://localhost:8000"
